# Silver Layer: Traffic Hazards Transformation

This notebook transforms raw traffic hazard data from Bronze to Silver layer.

## Transformations Applied:
- **Column Standardization**: All columns converted to snake_case
- **Hazard Type Standardization**: Clean and categorize hazard types (incident, roadwork, fire, flood, majorevent, alpine)
- **Timestamp Parsing**: Parse start_time, end_time, created, last_updated timestamps
- **Location Validation**: Ensure coordinates are within valid ranges and NSW bounds
- **Severity Classification**: Classify hazards by impact and is_major flag
- **Active Status**: Flag currently active hazards based on start/end times
- **Deduplication**: Remove duplicate records based on hazard_id

## Data Quality Validations:
- Count of records with invalid locations
- Count of records with missing critical fields
- Hazard type distribution
- Active vs expired hazards
- Major hazards count

In [0]:
# Configuration
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Table paths
BRONZE_TABLE = "mobility_ai.bronze.traffic_hazards_raw"
SILVER_TABLE = "mobility_ai.silver.traffic_hazards"

# NSW approximate bounds for location validation
NSW_LAT_MIN, NSW_LAT_MAX = -37.5, -28.0
NSW_LON_MIN, NSW_LON_MAX = 140.0, 154.0

print(f"Source: {BRONZE_TABLE}")
print(f"Target: {SILVER_TABLE}")

In [0]:
# Load bronze data
bronze_df = spark.table(BRONZE_TABLE)

print(f"Bronze record count: {bronze_df.count()}")

# Standardize column names to snake_case
def to_snake_case(col_name):
    import re
    s1 = re.sub('(.)([A-Z][a-z]+)', r'\1_\2', col_name)
    s2 = re.sub('([a-z0-9])([A-Z])', r'\1_\2', s1)
    return re.sub(r'[\s-]+', '_', s2).lower()

# Rename all columns
for col in bronze_df.columns:
    bronze_df = bronze_df.withColumnRenamed(col, to_snake_case(col))

# Parse timestamp fields (they come as ISO strings from API)
silver_df = bronze_df.withColumn(
    "start_time_ts",
    F.to_timestamp(F.col("start_time"))
).withColumn(
    "end_time_ts",
    F.to_timestamp(F.col("end_time"))
).withColumn(
    "created_ts",
    F.to_timestamp(F.col("created"))
).withColumn(
    "last_updated_ts",
    F.to_timestamp(F.col("last_updated"))
)

# Standardize hazard type (trim, lowercase)
silver_df = silver_df.withColumn(
    "hazard_type_clean",
    F.lower(F.trim(F.col("hazard_type")))
)

# Categorize hazard severity
silver_df = silver_df.withColumn(
    "severity",
    F.when(F.col("is_major") == True, "Major")
     .when(F.col("impact") == True, "Moderate")
     .otherwise("Minor")
)

# Calculate if hazard is currently active
silver_df = silver_df.withColumn(
    "is_active",
    F.when(
        (F.col("start_time_ts").isNotNull()) &
        (F.col("start_time_ts") <= F.current_timestamp()) &
        ((F.col("end_time_ts").isNull()) | (F.col("end_time_ts") >= F.current_timestamp())),
        True
    ).otherwise(False)
)

# Add data quality flags
silver_df = silver_df.withColumn(
    "has_valid_location",
    F.when(
        (F.col("latitude").isNotNull()) & 
        (F.col("longitude").isNotNull()) &
        (F.col("latitude").between(NSW_LAT_MIN, NSW_LAT_MAX)) &
        (F.col("longitude").between(NSW_LON_MIN, NSW_LON_MAX)),
        True
    ).otherwise(False)
)

silver_df = silver_df.withColumn(
    "has_valid_hazard_id",
    F.when(
        (F.col("hazard_id").isNotNull()) & 
        (F.length(F.col("hazard_id")) > 0),
        True
    ).otherwise(False)
)

silver_df = silver_df.withColumn(
    "has_valid_headline",
    F.when(
        (F.col("headline").isNotNull()) & 
        (F.length(F.col("headline")) > 0),
        True
    ).otherwise(False)
)

silver_df = silver_df.withColumn(
    "has_valid_start_time",
    F.when(F.col("start_time_ts").isNotNull(), True)
     .otherwise(False)
)

# Deduplicate only records with valid hazard_id
# For records without hazard_id, use composite key of headline + location + created_ts
silver_df = silver_df.withColumn(
    "dedup_key",
    F.when(
        F.col("has_valid_hazard_id") == True,
        F.col("hazard_id")
    ).otherwise(
        F.concat_ws("|",
            F.coalesce(F.col("headline"), F.lit("")),
            F.coalesce(F.col("latitude").cast("string"), F.lit("")),
            F.coalesce(F.col("longitude").cast("string"), F.lit("")),
            F.coalesce(F.col("hazard_type_clean"), F.lit(""))
        )
    )
)

window_spec = Window.partitionBy("dedup_key").orderBy(F.col("last_updated_ts").desc_nulls_last())

silver_df = silver_df.withColumn("row_num", F.row_number().over(window_spec))
silver_df = silver_df.filter(F.col("row_num") == 1).drop("row_num", "dedup_key")

# Add processing metadata
silver_df = silver_df.withColumn("silver_processed_at", F.current_timestamp())

print(f"Silver record count after transformations: {silver_df.count()}")
display(silver_df.limit(10))

In [0]:
# Write to silver table
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(SILVER_TABLE)

print(f"✅ Successfully wrote {silver_df.count()} records to {SILVER_TABLE}")

In [0]:
# Validate silver data
validation_df = spark.table(SILVER_TABLE)

# Quality metrics
total_count = validation_df.count()

if total_count == 0:
    print("=" * 60)
    print("DATA QUALITY REPORT")
    print("=" * 60)
    print("⚠️  No records found in the traffic_hazards_raw bronze table")
    print("Total Records: 0")
    print("=" * 60)
else:
    invalid_location = validation_df.filter(F.col("has_valid_location") == False).count()
    missing_hazard_id = validation_df.filter(F.col("has_valid_hazard_id") == False).count()
    missing_headline = validation_df.filter(F.col("has_valid_headline") == False).count()
    missing_start_time = validation_df.filter(F.col("has_valid_start_time") == False).count()
    active_hazards = validation_df.filter(F.col("is_active") == True).count()
    major_hazards = validation_df.filter(F.col("is_major") == True).count()

    print("=" * 60)
    print("DATA QUALITY REPORT")
    print("=" * 60)
    print(f"Total Records: {total_count}")
    print(f"Invalid Location: {invalid_location} ({invalid_location/total_count*100:.2f}%)")
    print(f"Missing Hazard ID: {missing_hazard_id} ({missing_hazard_id/total_count*100:.2f}%)")
    print(f"Missing Headline: {missing_headline} ({missing_headline/total_count*100:.2f}%)")
    print(f"Missing Start Time: {missing_start_time} ({missing_start_time/total_count*100:.2f}%)")
    print(f"\nActive Hazards: {active_hazards} ({active_hazards/total_count*100:.2f}%)")
    print(f"Major Hazards: {major_hazards} ({major_hazards/total_count*100:.2f}%)")
    print("=" * 60)

    # Hazard type distribution
    print("\nHazard Type Distribution:")
    validation_df.groupBy("hazard_type_clean").count().orderBy("count", ascending=False).show()

    # Severity distribution
    print("\nSeverity Distribution:")
    validation_df.groupBy("severity").count().orderBy("severity").show()

    # Active vs Expired
    print("\nActive Status:")
    validation_df.groupBy("is_active").count().orderBy("is_active", ascending=False).show()

    # Main category distribution
    print("\nMain Category Distribution:")
    validation_df.groupBy("main_category").count().orderBy("count", ascending=False).show()

    # Show sample of flagged records
    print("\nSample of records with data quality issues:")
    validation_df.filter(
        (F.col("has_valid_location") == False) | 
        (F.col("has_valid_hazard_id") == False) |
        (F.col("has_valid_headline") == False) |
        (F.col("has_valid_start_time") == False)
    ).select(
        "hazard_id", "hazard_type_clean", "headline", "latitude", "longitude",
        "start_time_ts", "has_valid_location", "has_valid_hazard_id", "has_valid_headline", "has_valid_start_time"
    ).show(10, truncate=False)